# <font color="steelblue">13. Relación entre variables cuantitativas: correlación y regresión</font>

*Unidad 2. Estadística descriptiva* · Sesión teórica 13 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`municipios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["municipios.csv"]
PREGUNTAS = ["q_P13_1.json", "q_P13_2.json", "q_P13_3.json", "q_P13_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«Los municipios con más renta tienen más hogares conectados a internet.» «Donde más se lee la prensa, mayor es la edad media.» Las dos frases describen **correlaciones**. Este cuaderno enseña a medirlas, a dibujar la recta que las resume y, sobre todo, a no confundir correlación con causa.

**Qué necesitas saber antes**: los cuadernos 8 y 10 (media y desviación típica).

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Dibujar e interpretar un diagrama de dispersión.
2. Calcular e interpretar el coeficiente de correlación de Pearson.
3. Ajustar una recta de regresión y usarla para describir la relación.
4. Explicar por qué correlación no implica causalidad.

## <font color='steelblue'>Contenidos</font>

1. El diagrama de dispersión
2. El coeficiente de correlación
3. La recta de regresión
4. Correlación no es causalidad

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. El diagrama de dispersión</font>

In [ ]:
muni = pd.read_csv("municipios.csv")
muni.head()

Cada punto es un municipio: su renta en el eje horizontal y el porcentaje de hogares con internet en el vertical.

In [ ]:
plt.scatter(muni["renta_media_persona"], muni["hogares_internet_pct"], alpha=0.6, color="steelblue")
plt.xlabel("Renta media por persona (€)")
plt.ylabel("Hogares con internet (%)")
plt.title("Renta e internet en 140 municipios (datos simulados)")
plt.show()

**Cómo leerlo**: la nube sube de izquierda a derecha: los municipios con más renta tienden a tener más hogares conectados. Es una relación **positiva** y aproximadamente **lineal**.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P13_1.json")

## <font color='steelblue'>2. El coeficiente de correlación</font>

El **coeficiente de correlación de Pearson** ($r$) mide la fuerza de la relación **lineal**. Va de −1 a 1:

| $r$ | Lectura |
| :-- | :-- |
| cerca de 1 | relación positiva fuerte |
| cerca de 0 | sin relación lineal |
| cerca de −1 | relación negativa fuerte |

In [ ]:
print("Renta - internet:", round(muni["renta_media_persona"].corr(muni["hogares_internet_pct"]), 2))
print("Renta - paro:    ", round(muni["renta_media_persona"].corr(muni["tasa_paro"]), 2))
cols = ["renta_media_persona", "tasa_paro", "edad_media", "hogares_internet_pct", "lectores_prensa_pct"]
muni[cols].corr().round(2)

**Cómo leerlo**: la renta se relaciona positivamente con internet y negativamente con el paro. La edad media y los lectores de prensa tienen la correlación más fuerte de la tabla.

⚠️ $r$ solo mide relaciones **lineales**. Una relación en forma de U puede dar $r \approx 0$. **Mira siempre el gráfico.**

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P13_2.json")

## <font color='steelblue'>3. La recta de regresión</font>

La **recta de regresión** es la recta que mejor resume la nube: $y = a + b\,x$. La **pendiente** $b$ dice cuánto cambia $y$, en promedio, cuando $x$ aumenta una unidad.

In [ ]:
x = muni["renta_media_persona"]
y = muni["hogares_internet_pct"]
b, a = np.polyfit(x, y, 1)
print(f"Recta: internet = {a:.1f} + {b:.5f} × renta")
print(f"Por cada 1.000 € más de renta, {b * 1000:.1f} puntos más de hogares con internet")

In [ ]:
plt.scatter(x, y, alpha=0.5, color="steelblue")
xs = np.linspace(x.min(), x.max(), 100)
plt.plot(xs, a + b * xs, color="firebrick", label="Recta de regresión")
plt.xlabel("Renta media por persona (€)")
plt.ylabel("Hogares con internet (%)")
plt.legend()
plt.show()

El **coeficiente de determinación** $R^2 = r^2$ indica qué parte de la variación de $y$ «se explica» por la recta. Si $r = 0{,}64$, entonces $R^2 \approx 0{,}41$: la renta da cuenta de un 41 % de las diferencias de conexión entre municipios; el resto depende de otras cosas.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P13_3.json")

## <font color='steelblue'>4. Correlación no es causalidad</font>

Los municipios con más lectores de prensa tienen más edad media. ¿Leer prensa envejece? Evidentemente no: la gente mayor lee más prensa. Y otras correlaciones esconden una **tercera variable** que mueve a las dos (una **variable de confusión**).

Antes de escribir que A causa B, pregúntate:
1. ¿Podría ser al revés (B causa A)?
2. ¿Hay una tercera variable que explique las dos?
3. ¿Es casualidad? (con muchos cruces, alguno sale por azar)

Solo un **experimento** o un diseño cuidadoso permite hablar de causas.

In [ ]:
plt.scatter(muni["edad_media"], muni["lectores_prensa_pct"], alpha=0.6, color="darkorange")
plt.xlabel("Edad media (años)")
plt.ylabel("Lectores de prensa (%)")
plt.title(f"r = {muni['edad_media'].corr(muni['lectores_prensa_pct']):.2f}: ¿la prensa envejece?")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y marca **todas** las respuestas correctas:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P13_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Dibuja el diagrama de dispersión entre la renta media y la tasa de paro.

<details><summary>Ver solución</summary>

```python
plt.scatter(muni["renta_media_persona"], muni["tasa_paro"], alpha=0.6)
plt.xlabel("Renta"); plt.ylabel("Paro (%)")
plt.show()
```
</details>

### **Ejercicio 2** *(sección 2)*

¿Qué pareja de variables de la tabla tiene la correlación más cercana a cero?

<details><summary>Ver solución</summary>

```python
muni[cols].corr().round(2)
```
</details>

### **Ejercicio 3** *(sección 3)*

Ajusta la recta del paro en función de la renta. ¿Cuánto baja el paro, en promedio, por cada 1.000 € más de renta?

<details><summary>Ver solución</summary>

```python
b, a = np.polyfit(muni["renta_media_persona"], muni["tasa_paro"], 1)
b * 1000
```
</details>

### **Ejercicio 4** *(sección 3)*

Calcula el R² de esa recta.

<details><summary>Ver solución</summary>

```python
muni["renta_media_persona"].corr(muni["tasa_paro"]) ** 2
```
</details>

### **Ejercicio 5** *(sección 4)*

Escribe un titular honesto y otro engañoso con la correlación entre edad media y lectores de prensa.

<details><summary>Ver idea clave</summary>

Honesto: «Los municipios más envejecidos son los que más leen prensa». Engañoso: «Leer prensa hace que un municipio envejezca».
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- El **diagrama de dispersión** es el primer paso para estudiar dos variables cuantitativas.
- El **coeficiente de correlación** $r$ (de −1 a 1) mide la fuerza de la relación **lineal**; mira siempre el gráfico.
- La **recta de regresión** ($y = a + b\,x$) resume la relación; la pendiente $b$ es el cambio medio de $y$ por unidad de $x$.
- $R^2 = r^2$ es la parte de la variación de $y$ asociada a $x$.
- **Correlación no implica causalidad**: piensa en la causa inversa, las variables de confusión y el azar.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Diagrama de dispersión** | Gráfico de puntos de dos variables cuantitativas. |
| **Correlación de Pearson** | Medida de la relación lineal entre dos variables, de −1 a 1. |
| **Recta de regresión** | Recta que mejor resume la relación lineal. |
| **Variable de confusión** | Tercera variable que influye en las dos que se comparan. |
| **R²** | Proporción de la variación explicada por la recta. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `np.polyfit` da error con NaN | Hay datos que faltan | Elimina antes las filas incompletas con `dropna()` |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

Con esto termina la unidad 2. En el cuaderno 14 empieza la unidad de números índice: tasas de variación e índices simples.

**Siguiente cuaderno**: [14. Tasas de variación e índices simples](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P14_tasas_de_variacion_e_indices_simples.ipynb)